<a href="https://colab.research.google.com/github/daviminati/Trab_Final_IA/blob/main/Sprint_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 2: Processamento de Dados de Texto para LLMs
**Disciplina / Projeto Integrador LLM**

Este notebook contempla a implementação do pipeline completo do **Capítulo 2 do livro *Build a Large Language Model (From Scratch)***, convertendo texto bruto em tensores de entrada e alvos para o treinamento de um modelo de linguagem.

**Pipeline implementado:**
`Texto Bruto` → `Tokenização` → `Token IDs` → `Janela Deslizante (X, Y)` → `DataLoader` → `Embeddings (Token + Posicional)`

In [6]:
# 1. Instalação de dependências e importação de bibliotecas
!pip install tiktoken torch -q

import re
import os
import urllib.request
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

# Semente para reprodutibilidade dos experimentos
torch.manual_seed(42)
print("Bibliotecas inicializadas com sucesso. Versão do PyTorch:", torch.__version__)

Bibliotecas inicializadas com sucesso. Versão do PyTorch: 2.11.0+cpu


## 3.1 e 3.2 Tokenização, Vocabulário e Token IDs

Nesta etapa, implementamos:
1. Um **Tokenizador básico via Expressões Regulares** para entender o mapeamento manual entre palavras, dicionário (vocabulário) e inteiros.
2. O **Tokenizador BPE (Byte Pair Encoding)** utilizando a biblioteca `tiktoken` com o vocabulário oficial do GPT-2.

In [7]:
# Implementação do Tokenizador Manual (Regex + Vocabulário)
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s+)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>"
            for item in preprocessed
        ]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

# Teste com texto em português e token especial
vocab_exemplo = {"Desenvolver": 0, "um": 1, "LLM": 2, "é": 3, "incrível": 4, ".": 5, "<|unk|>": 6, "<|endoftext|>": 7}
tokenizer_simples = SimpleTokenizerV2(vocab_exemplo)

texto_teste = "Desenvolver um LLM do zero é incrível. <|endoftext|>"
ids_simples = tokenizer_simples.encode(texto_teste)
print("--- Tokenizador Manual ---")
print("IDs gerados:", ids_simples)
print("Texto reconstruído:", tokenizer_simples.decode(ids_simples))

# Comparação com BPE (Tiktoken)
bpe_tokenizer = tiktoken.get_encoding("gpt2")
ids_bpe = bpe_tokenizer.encode(texto_teste, allowed_special={"<|endoftext|>"})

print("\n--- Tokenizador BPE (GPT-2 / Tiktoken) ---")
print("Quantidade de Token IDs (BPE):", len(ids_bpe))
print("IDs gerados (BPE):", ids_bpe)
print("Texto reconstruído (BPE):", bpe_tokenizer.decode(ids_bpe))

--- Tokenizador Manual ---
IDs gerados: [0, 1, 2, 6, 6, 3, 4, 5, 7]
Texto reconstruído: Desenvolver um LLM <|unk|> <|unk|> é incrível. <|endoftext|>

--- Tokenizador BPE (GPT-2 / Tiktoken) ---
Quantidade de Token IDs (BPE): 17
IDs gerados (BPE): [5960, 268, 10396, 332, 23781, 27140, 44, 466, 6632, 38251, 753, 81, 8836, 626, 13, 220, 50256]
Texto reconstruído (BPE): Desenvolver um LLM do zero é incrível. <|endoftext|>


## 3.3 e 3.6 Preparação de Sequências e Organização dos Dados

Utilizamos o texto do conto *"The Verdict"* (referência do livro) para construir o `Dataset` com a técnica de **Janela Deslizante (Sliding Window)**. Essa técnica cria os pares de treinamento auto-supervisionados:
* **Entrada ($X$):** Sequência de tokens até o tamanho de contexto (`max_length`).
* **Alvo ($Y$):** A mesma sequência deslocada de 1 posição para a direita (próximo token previsto).

In [8]:
# Download do arquivo de texto do livro
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
file_path = "the-verdict.txt"

if not os.path.exists(file_path):
    urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Texto carregado! Total de caracteres: {len(raw_text)}")

# Classe Dataset com Janela Deslizante
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

# Teste de extração de lotes
loader_teste = create_dataloader_v1(raw_text, batch_size=2, max_length=4, stride=1, shuffle=False)
inputs_x, targets_y = next(iter(loader_teste))

print("\n--- Amostra de Lote (Batch Size=2, Context Length=4) ---")
print("Entrada (X):\n", inputs_x)
print("Alvo (Y):\n", targets_y)

Texto carregado! Total de caracteres: 20479

--- Amostra de Lote (Batch Size=2, Context Length=4) ---
Entrada (X):
 tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]])
Alvo (Y):
 tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]])
